# Week 3 - Unsupervised Learning + Model Evaluation
**Skill Nexis - Machine Learning & AI Internship**

Name: Mohit  
Roll No: 24SCSE1420017

## Setup (Colab + Kaggle)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!pip install kaggle
!mkdir -p /root/.kaggle
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d uciml/iris
!unzip -o iris.zip

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import pickle
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score, silhouette_score

## Load the Iris dataset

In [ ]:
iris = pd.read_csv('Iris.csv')
iris.head()

In [ ]:
iris.info()

In [ ]:
iris['Species'].value_counts()

In [ ]:
X = iris.drop(['Id', 'Species'], axis=1)
X.describe()

## Assignment 1 - K-Means on Iris and visualize clusters

Elbow method first, to check which k to use.

In [ ]:
inertia = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertia.append(km.inertia_)

plt.plot(range(1, 11), inertia, marker='o')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow method')
plt.show()

The curve bends at k = 3, which also matches the 3 species in the dataset.

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans.fit(X)
iris['cluster'] = kmeans.labels_
iris['cluster'].value_counts()

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(X['PetalLengthCm'], X['PetalWidthCm'], c=kmeans.labels_, cmap='viridis')
plt.scatter(kmeans.cluster_centers_[:, 2], kmeans.cluster_centers_[:, 3], c='red', marker='X', s=200, label='centers')
plt.xlabel('Petal Length (cm)')
plt.ylabel('Petal Width (cm)')
plt.title('K-Means clusters (k=3)')
plt.legend()
plt.show()

## Assignment 2 - PCA to reduce dataset dimensions

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print('before:', X.shape)
print('after :', X_pca.shape)

In [ ]:
species_codes = iris['Species'].astype('category').cat.codes
plt.figure(figsize=(7, 5))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=species_codes, cmap='viridis')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Iris after PCA (4D to 2D)')
plt.show()

In [ ]:
print(pca.explained_variance_ratio_)
print('total:', pca.explained_variance_ratio_.sum())

## Mini Project 3 - Iris Flower Clustering
Dataset: Iris (UCI / Kaggle)

Tasks: apply K-Means (k=3), visualize clusters, compare predicted clusters vs true labels.

### Apply K-Means (k=3)

In [ ]:
km = KMeans(n_clusters=3, random_state=42, n_init=10)
iris['cluster'] = km.fit_predict(X)
iris[['Species', 'cluster']].head(10)

### Visualize clusters

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))

ax[0].scatter(X['SepalLengthCm'], X['SepalWidthCm'], c=iris['cluster'], cmap='viridis')
ax[0].set_xlabel('Sepal Length (cm)')
ax[0].set_ylabel('Sepal Width (cm)')
ax[0].set_title('Sepal')

ax[1].scatter(X['PetalLengthCm'], X['PetalWidthCm'], c=iris['cluster'], cmap='viridis')
ax[1].scatter(km.cluster_centers_[:, 2], km.cluster_centers_[:, 3], c='red', marker='X', s=200)
ax[1].set_xlabel('Petal Length (cm)')
ax[1].set_ylabel('Petal Width (cm)')
ax[1].set_title('Petal')

plt.show()

### Predicted clusters vs true labels

In [ ]:
pd.crosstab(iris['Species'], iris['cluster'])

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
codes = iris['Species'].astype('category').cat.codes

ax[0].scatter(X['PetalLengthCm'], X['PetalWidthCm'], c=codes, cmap='viridis')
ax[0].set_title('True species')

ax[1].scatter(X['PetalLengthCm'], X['PetalWidthCm'], c=iris['cluster'], cmap='viridis')
ax[1].set_title('K-Means clusters')

for a in ax:
    a.set_xlabel('Petal Length (cm)')
    a.set_ylabel('Petal Width (cm)')
plt.show()

In [ ]:
mapping = {}
for c in range(3):
    mapping[c] = iris.loc[iris['cluster'] == c, 'Species'].mode()[0]

iris['cluster_species'] = iris['cluster'].map(mapping)
print(mapping)
print('Match with true labels:', (iris['cluster_species'] == iris['Species']).mean())
print('Adjusted Rand score   :', adjusted_rand_score(iris['Species'], iris['cluster']))
print('Silhouette score      :', silhouette_score(X, iris['cluster']))

Iris-setosa gets its own cluster, but Iris-versicolor and Iris-virginica overlap, so some of them end up in the wrong cluster. K-Means does not use the labels, so it cannot be perfect here.

## Save and load the model (joblib and pickle)

In [ ]:
joblib.dump(km, 'kmeans_model.joblib')

with open('kmeans_model.pkl', 'wb') as f:
    pickle.dump(km, f)

In [ ]:
model_j = joblib.load('kmeans_model.joblib')

with open('kmeans_model.pkl', 'rb') as f:
    model_p = pickle.load(f)

sample = pd.DataFrame([[5.1, 3.5, 1.4, 0.2]], columns=X.columns)
print(model_j.predict(sample), model_p.predict(sample))

In [ ]:
!cp kmeans_model.joblib kmeans_model.pkl /content/drive/MyDrive/

## Practice Set - Week 3

**Q1. Apply K-Means on the Iris dataset and print cluster centers**

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans.fit(X)
print(kmeans.cluster_centers_)

In [ ]:
pd.DataFrame(kmeans.cluster_centers_, columns=X.columns)

**Q2. Visualize clusters with matplotlib**

In [ ]:
colors = ['red', 'green', 'blue']
for i in range(3):
    plt.scatter(X.loc[kmeans.labels_ == i, 'PetalLengthCm'], X.loc[kmeans.labels_ == i, 'PetalWidthCm'],
                c=colors[i], label='Cluster ' + str(i))
plt.scatter(kmeans.cluster_centers_[:, 2], kmeans.cluster_centers_[:, 3], c='black', marker='X', s=200, label='Centers')
plt.xlabel('Petal Length (cm)')
plt.ylabel('Petal Width (cm)')
plt.legend()
plt.show()

**Q3. Apply PCA to reduce the dataset to 2D**

In [ ]:
pca2 = PCA(n_components=2)
X_2d = pca2.fit_transform(X_scaled)
print(X_2d.shape)
pd.DataFrame(X_2d, columns=['PC1', 'PC2']).head()

In [ ]:
plt.scatter(X_2d[:, 0], X_2d[:, 1], c=kmeans.labels_, cmap='viridis')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('K-Means clusters on the PCA 2D data')
plt.show()

**Q4. Find the explained variance ratio**

In [ ]:
print('Explained variance ratio:', pca2.explained_variance_ratio_)
print('Total variance kept      :', pca2.explained_variance_ratio_.sum())

In [ ]:
pca_full = PCA()
pca_full.fit(X_scaled)
plt.plot(range(1, 5), np.cumsum(pca_full.explained_variance_ratio_), marker='o')
plt.xlabel('Number of components')
plt.ylabel('Cumulative explained variance')
plt.show()

The first two components keep most of the information (the exact value is printed above), so going from 4 features to 2 loses very little.